In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here

# Import required libraries
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

# By using Custom Dataset class
class PotatoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Step 1: Get all class names (folder names)
        self.classes = sorted(os.listdir(root_dir)) # orders alphabitacally

        # Step 2: Create a mapping from class name to integer label
        self.class_to_idx = {class_name: i for i, class_name in enumerate(self.classes)}

        # Step 3: Collect all image paths and their labels
        self.image_paths = []
        self.labels = []

        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            class_idx = self.class_to_idx[class_name]

            # Get all image files in this class folder
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                self.image_paths.append(img_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')  # Ensure RGB format

        # Get label
        label = self.labels[idx]

        # Apply transforms if provided
        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets for train, validation, and test
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset = PotatoDataset(
    root_dir=os.path.join(path, 'PlantVillage', "train"),
    transform=transform
)


test_dataset = PotatoDataset(
    root_dir=os.path.join(path, 'PlantVillage', "test"),
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"Classes: {train_dataset.classes[:5]}...")  # Show first 5 classes

# Display Images
import random
import numpy as np
import glob

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):

    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        # check if    this  is this type  (in this case check if img is a tensor)
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Visualize training samples
visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")

In [ ]:
class SimpleFashionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(

            #  Conv
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 32x32
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                         # 16 x 16

            nn.Conv2d(16, 32, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 8 x 8

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                        #  4 x 4

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                        # 2 x 2

            nn.Conv2d(128, 256, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),             #        1 x 1

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(256 * 1 * 1 , 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 3) # 3 classes
        )

    def forward(self, x):
        # TO-DO: Pass x through features
        x = self.features(x)
        # TO-DO: Pass result through classifier
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here
# TO DO
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
#  For BC  # images, masks = images.to(device), masks.to(device).to(torch.float)
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

        outputs = model(images)
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
# For BC          images, masks = images.to(device), masks.to(device).to(torch.float)
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# Write your code here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
model = SimpleFashionCNN().to(device)


criterion = nn.CrossEntropyLoss()


optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 10

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = validate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {test_loss:.4f}, Val Acc: {test_acc:.4f}')


plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"], label = "Train Loss", marker='o') ## , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "Validation Loss", marker='o')## , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"], label="Validation Accuracy",  marker='o')## , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()##

In [ ]:
# Write your code here

class ResidualCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(

            #  Conv
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 32x32
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                         # 16 x 16

            nn.Conv2d(16, 32, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 8 x 8

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                        #  4 x 4
# Here
            conc = torch.cat([nn.Conv2d(32, 64, kernel_size = 3, padding = 1), nn.Conv2d(1, 16, kernel_size=3, padding=1)], dim=1)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                        # 2 x 2

            nn.Conv2d(128, 256, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),             #        1 x 1

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(256 * 1 * 1 , 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 3) # 3 classes
        )

    def forward(self, x):
        # TO-DO: Pass x through features
        x = self.features(x)
        # TO-DO: Pass result through classifier
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
model = ResidualCNN().to(device)


criterion = nn.CrossEntropyLoss()


optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 10

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = validate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {test_loss:.4f}, Val Acc: {test_acc:.4f}')


plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"], label = "Train Loss", marker='o') ## , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "Validation Loss", marker='o')## , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"], label="Validation Accuracy",  marker='o')## , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()##
